# Soup take-home — v2 on **Kaggle**: one clean run + the four open items from v1

**Before running (Kaggle editor, right-hand panel → Settings):**
1. **Accelerator: GPU T4 x2.** The notebook pins ONE T4 (see cell 1). A P100 is refused, because v1 was measured on a T4.
2. **Internet: On.** It is needed to install Soup and download Qwen2.5-1.5B-Instruct (Kaggle asks for phone verification once).
3. **Add Input → your dataset `SOuptsk`.** It must hold the v2 project: either `soup_task_v2.zip` or its unpacked files.
   Cell 1 searches `/kaggle/input` for it, so the exact folder layout does not matter.

**Run:** *Save Version → Save & Run All (Commit)* runs it headless, with no idle timeout, in ~2 h.
`/kaggle/working/deliverables_v2.zip` then appears under the version's **Output**. *Run All* in the editor works too,
but keep the tab active. **If a cell stops with `STOP:`, send that output; every cell refuses to continue on
missing or mismatched artifacts.**

What v2 changes vs v1: (1) a clean single pass, with torchao removed first, no retraining, and SHA-256 provenance
re-checked on every step; (2) Soup's general suites with every answer recorded, in fp16; (3) the allocator's peak
reserved memory, the CUDA context and the GPU activation measurement; (4) the grammar guard measured, with
opposite-action errors counted. The 4-bit gate in `verify.py` now covers every eval set.

Kaggle-only changes (nothing in the experiment): paths `/content` → `/kaggle/working`; the project is read from
`/kaggle/input`; GPU 0 only; nvidia-smi traces GPU 0; no `google.colab`; the result is a file in `/kaggle/working`.

In [ ]:
import os, subprocess, time, datetime, json, hashlib, glob, zipfile, shutil

# --- ONE GPU. Kaggle's "T4 x2" exposes two GPUs, and a single-process HF Trainer that sees two
# wraps the model in nn.DataParallel. That would double the batch (8 -> 8 per GPU x 2) and change
# the training. It must be set before anything initialises CUDA; every subprocess inherits it.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"
UTC = datetime.timezone.utc

INPUT, WORK = "/kaggle/input", "/kaggle/working"
PROJECT = f"{WORK}/soup_task"                 # was /content/soup_task on Colab
REQUIRED = ["build_data.py", "verify.py", "train_with_stats.py", "general_probe.py", "guarded_eval.py",
            "silent_failures.py", "quantize_nf4.py", "memory_budget.py", "measure_activations.py",
            "phase_report.py", "configs/main.yaml", "configs/control.yaml", "data/manifest.json",
            "data/test_real.tsv", "data/test_novel.jsonl"]

def is_project(d):
    return all(os.path.exists(os.path.join(d, r)) for r in REQUIRED)

def tree(root, depth=4):
    for dirpath, dirs, files in os.walk(root, followlinks=True):
        lvl = dirpath[len(root):].count(os.sep)
        if lvl >= depth: dirs[:] = []; continue
        print("  " * lvl + os.path.basename(dirpath) + "/")
        for f in sorted(files)[:12]: print("  " * (lvl + 1) + f)
        if len(files) > 12: print("  " * (lvl + 1) + f"... {len(files) - 12} more")

def locate_project():
    """Find the v2 project in /kaggle/input WITHOUT assuming its layout: an unpacked folder (Kaggle unpacks an
    uploaded zip), or a zip somewhere inside the dataset. /kaggle/input is read-only, so it is copied to
    /kaggle/working first."""
    if is_project(PROJECT):
        return f"already present in {PROJECT}"
    print("Attached inputs under /kaggle/input:"); tree(INPUT)
    # prefer a dataset whose path mentions SOuptsk (case-insensitive), then anything else
    found = []
    for dirpath, dirs, files in os.walk(INPUT, followlinks=True):
        if is_project(dirpath): found.append(("dir", dirpath))
        for f in files:
            if f.lower().endswith(".zip"): found.append(("zip", os.path.join(dirpath, f)))
    found.sort(key=lambda x: ("souptsk" not in x[1].lower(), x[0] != "dir", x[1]))
    for kind, path in found:
        if kind == "dir":
            # copyfile, not copy2: /kaggle/input is read-only and its mode bits must not follow, or
            # build_data.py could not rewrite data/*.jsonl
            shutil.copytree(path, PROJECT, copy_function=shutil.copyfile)
            subprocess.run(["chmod", "-R", "u+rwX", PROJECT], check=True)
            return f"copied unpacked project {path} -> {PROJECT}"
        with zipfile.ZipFile(path) as z:
            names = z.namelist()
            if not any(n.endswith("verify.py") for n in names): continue
            tmp = f"{WORK}/_unzip"; shutil.rmtree(tmp, ignore_errors=True)
            z.extractall(tmp)                                   # zipfile drops absolute / '..' parts
        for dirpath, dirs, files in os.walk(tmp):
            if is_project(dirpath):
                shutil.move(dirpath, PROJECT); shutil.rmtree(tmp, ignore_errors=True)
                return f"extracted {path} -> {PROJECT}"
        raise RuntimeError(f"STOP: {path} contains verify.py but not the full v2 project "
                           f"(needs {REQUIRED}). Is it the v1 soup_task.zip? Upload soup_task_v2.zip.")
    raise RuntimeError("STOP: no v2 project found under /kaggle/input (tree above). In the editor: Add Input -> "
                       "your dataset SOuptsk, which must contain soup_task_v2.zip or its unpacked files.")

print(locate_project())
os.chdir(PROJECT)
for d in ("logs", "results", "runs"):                     # checkpoints and models are written under runs/
    os.makedirs(d, exist_ok=True)
print("working directory:", os.getcwd())

def now(): return datetime.datetime.now(UTC).strftime('%Y-%m-%dT%H:%M:%S.%f')[:-3] + "Z"

def sh(phase, cmd, check=True):
    """Run cmd, timestamp every line, tee to logs/<phase>.log, mark phase start/end. Fails loudly by default."""
    with open("logs/phases.csv", "a") as f: f.write(f"start,{phase},{time.time()}\n")
    t0 = time.time()
    with open(f"logs/{phase}.log", "a") as log:
        log.write(f"### {now()} $ {cmd}\n")
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            stamped = f"[{now()}] {line.rstrip()}"
            print(stamped); log.write(stamped + "\n")
        rc = p.wait()
        log.write(f"### exit code {rc} after {time.time()-t0:.1f}s\n")
    with open("logs/phases.csv", "a") as f: f.write(f"end,{phase},{time.time()}\n")
    print(f"--- {phase}: exit {rc}, {time.time()-t0:.0f}s")
    if check and rc != 0: raise RuntimeError(f"STOP: {phase} failed (exit {rc}). Send the lines above; do not run further cells.")
    return rc

def need(*paths):
    missing = [p for p in paths if not os.path.exists(p)]
    if missing: raise RuntimeError(f"STOP: missing {missing} - an earlier cell did not finish. Do not continue.")

def hashes(d):
    return {os.path.basename(f): hashlib.sha256(open(f, "rb").read()).hexdigest()[:16]
            for f in sorted(glob.glob(f"{d}/*.safetensors"))}

PROV = "results/provenance.json"
def record(name, d):
    p = json.load(open(PROV)) if os.path.exists(PROV) else {}
    p[name] = {"hashes": hashes(d), "recorded_at": now()}
    json.dump(p, open(PROV, "w"), indent=1); print("provenance", name, p[name])

def check_same(name, d):
    """Refuse to evaluate an artifact whose bytes differ from the ones recorded when it was made."""
    rec = json.load(open(PROV))[name]["hashes"]
    cur = hashes(d)
    if cur != rec: raise RuntimeError(f"STOP: {d} changed since it was recorded ({rec} -> {cur}).")
    print(f"provenance ok: {name} {cur}")

subprocess.Popen("nvidia-smi -i 0 --query-gpu=timestamp,memory.used,memory.total,utilization.gpu,power.draw "
                 "--format=csv,nounits -lms 200 > logs/nvidia_smi_trace.csv", shell=True)
subprocess.Popen("while true; do echo \"===== $(date -u +%FT%TZ)\"; nvidia-smi; sleep 30; done > logs/nvidia_smi_raw.log 2>&1", shell=True)
sh("env_gpu", "nvidia-smi; nvidia-smi --query-gpu=index,name,driver_version,memory.total,compute_cap --format=csv")

## 0. Pre-flight (Kaggle): exactly one T4 visible, internet on, enough disk

In [ ]:
q = subprocess.run("nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv,noheader",
                   shell=True, capture_output=True, text=True).stdout.strip().splitlines()
print("GPUs on the machine:", q)
if not q:
    raise RuntimeError("STOP: no GPU. Settings -> Accelerator -> GPU T4 x2.")
if "T4" not in q[0]:
    raise RuntimeError(f"STOP: GPU 0 is {q[0]}. v1 was measured on a Tesla T4, so memory, speed and fp16/NF4 "
                       "numerics are only comparable on a T4. Settings -> Accelerator -> GPU T4 x2.")
vis = subprocess.run("python -c \"import torch;print(torch.cuda.device_count(), torch.cuda.get_device_name(0))\"",
                     shell=True, capture_output=True, text=True)
print("visible to PyTorch:", vis.stdout.strip() or vis.stderr[-300:])
if not vis.stdout.startswith("1 "):
    raise RuntimeError("STOP: PyTorch must see exactly 1 GPU (CUDA_VISIBLE_DEVICES=0 is set in cell 1). "
                       "Restart the session and run from the top.")
import urllib.request
for url in ("https://pypi.org/simple/soup-cli/", "https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct"):
    try:
        urllib.request.urlopen(url, timeout=15); print("reachable:", url)
    except Exception as e:
        raise RuntimeError(f"STOP: {url} unreachable ({e}). Settings -> Internet -> On.")
for p in (WORK, os.path.expanduser("~")):
    free = shutil.disk_usage(p).free / 2**30
    print(f"free disk {p}: {free:.1f} GiB")
    if free < 12: raise RuntimeError(f"STOP: {p} has {free:.1f} GiB free; the run writes about 8 GiB of models.")

## 0b. Environment: torchao removed before anything imports peft (v1's failure)

In [ ]:
# Kaggle preinstalls torch/transformers/peft/accelerate. pip only replaces what Soup 0.77.2's pins require
# (transformers>=5.16.1,<6, peft>=0.20,<1, trl>=0.29,<1, torch>=2.6). Torch is recorded before and after so a
# swap is visible in the log. The long pip output goes to logs/pip_install.txt; its last lines are shown.
sh("env_torch_before", "python -c \"import torch;print('torch before', torch.__version__, torch.version.cuda)\"", check=False)
sh("env_install", 'pip install -q "soup-cli[train]==0.77.2" > logs/pip_install.txt 2>&1; rc=$?; tail -5 logs/pip_install.txt; '
                  'pip uninstall -y -q torchao 2>/dev/null; '
                  '[ -x /usr/bin/time ] || (apt-get -qq update > /dev/null && apt-get -qq install -y time > /dev/null) || true; exit $rc')
sh("env_versions", "which soup && python -c \"import importlib.util as u; assert u.find_spec('torchao') is None, 'torchao still installed'; "
                   "import torch,transformers,peft,trl,bitsandbytes as b;print('torch',torch.__version__,'cuda',torch.version.cuda,"
                   "'gpus',torch.cuda.device_count(),'bf16_supported',torch.cuda.is_bf16_supported());"
                   "print('transformers',transformers.__version__,'peft',peft.__version__,"
                   "'trl',trl.__version__,'bnb',b.__version__)\"; soup --version; pip freeze > logs/pip_freeze.txt")
sh("env_bnb_smoke", "python -c \"import torch,bitsandbytes.functional as F; w=torch.randn(64,64,device='cuda',dtype=torch.float16); "
                    "q,s=F.quantize_4bit(w,quant_type='nf4'); print('bnb nf4 ok, max err', (F.dequantize_4bit(q,s)-w).abs().max().item())\"")

## 1. Data — must rebuild byte-identical to the shipped manifest

In [ ]:
shipped = json.load(open("data/manifest.json"))["sha256"]
sh("data_build", "python build_data.py")
assert json.load(open("data/manifest.json"))["sha256"] == shipped, "STOP: data did not rebuild identically"
print("data reproducible: True")
sh("data_doctor", "soup --no-telemetry data doctor data/train.jsonl -m Qwen/Qwen2.5-1.5B-Instruct --max-length 256 --sample 1500 --show-mask 3 -o results/data_doctor.json", check=False)
sh("data_tokens", """python - <<'EOF'
import json, statistics
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
n = [len(tok(tok.apply_chat_template(json.loads(l)["messages"], tokenize=False), add_special_tokens=False)["input_ids"]) for l in open("data/train.jsonl")]
json.dump(n, open("data/tokens.json", "w"))
print("rows", len(n), "mean", statistics.mean(n), "p95", sorted(n)[int(.95*len(n))], "max", max(n))
EOF""")

## 2. Part 1 — memory budget before training, CUDA context, activations on the GPU

In [ ]:
sh("budget_before", "python memory_budget.py --config $(python -c \"from huggingface_hub import hf_hub_download as d;print(d('Qwen/Qwen2.5-1.5B-Instruct','config.json'))\") --tokens data/tokens.json --batch 8 --json-out results/memory_budget.json")
sh("budget_soup_profile", "soup --no-telemetry profile -c configs/main.yaml --json > results/soup_profile.json; cat results/soup_profile.json", check=False)
sh("budget_cuda_context", """python - <<'EOF'
import json, subprocess, torch
torch.zeros(1, device="cuda"); torch.cuda.synchronize()
used = float(subprocess.run(["nvidia-smi","--query-gpu=memory.used","--format=csv,noheader,nounits"],capture_output=True,text=True).stdout.split()[0])
ctx = {"nvidia_smi_MiB_with_1_tensor": used, "torch_reserved_MiB": torch.cuda.memory_reserved()/2**20}
ctx["context_MiB"] = used - ctx["torch_reserved_MiB"]
print(json.dumps(ctx)); json.dump(ctx, open("results/cuda_context.json","w"))
EOF""")
sh("budget_measure_activations", "python measure_activations.py --tokens 64 --batch 4 > results/activation_measurement.json; cat results/activation_measurement.json")

## 3. Train — main and control, in-process so the allocator counters are readable. Never overwrites an adapter.

In [ ]:
for name, cfg in (("main", "configs/main.yaml"), ("control", "configs/control.yaml")):
    if os.path.exists(f"runs/{name}/adapter_model.safetensors"):
        print(f"SKIP train_{name}: runs/{name} already has an adapter (delete the folder yourself to retrain)")
    else:
        sh(f"train_{name}", f"python train_with_stats.py {cfg} results/train_{name}_cuda_stats.json")
    need(f"runs/{name}/adapter_model.safetensors", f"runs/{name}/rewind.jsonl")
    p = json.load(open(PROV)) if os.path.exists(PROV) else {}
    if f"adapter_{name}" not in p: record(f"adapter_{name}", f"runs/{name}")
    else: check_same(f"adapter_{name}", f"runs/{name}")

## 4. Merge (fp16, CPU) and quantize (NF4)

In [ ]:
for name in ("main", "control"):
    check_same(f"adapter_{name}", f"runs/{name}")
    if not os.path.exists(f"runs/{name}-merged/config.json"):
        TIME = "/usr/bin/time -v " if os.path.exists("/usr/bin/time") else ""   # host-RAM peak of the merge
        sh(f"merge_{name}", f"{TIME}soup --no-telemetry merge --adapter runs/{name} --output runs/{name}-merged --dtype float16")
    need(f"runs/{name}-merged/config.json")
    if f"merged_{name}" not in json.load(open(PROV)): record(f"merged_{name}", f"runs/{name}-merged")
if not os.path.exists("runs/main-merged-nf4/config.json"):
    sh("quantize_main", "python quantize_nf4.py runs/main-merged runs/main-merged-nf4")
need("runs/main-merged-nf4/config.json")
if "nf4_main" not in json.load(open(PROV)): record("nf4_main", "runs/main-merged-nf4")
sh("artifact_sizes", "du -sh runs/main runs/main-merged runs/main-merged-nf4 runs/control runs/control-merged")

## 5. Part 2 — verify (provenance checked first) and the guarded re-score

In [ ]:
for n, d in (("adapter_main","runs/main"),("merged_main","runs/main-merged"),("nf4_main","runs/main-merged-nf4"),
             ("adapter_control","runs/control"),("merged_control","runs/control-merged")):
    check_same(n, d)
sh("verify", "python verify.py --base Qwen/Qwen2.5-1.5B-Instruct --adapter runs/main --merged runs/main-merged "
             "--quant4 runs/main-merged-nf4 --control-adapter runs/control --control-merged runs/control-merged "
             "--out results/verify.json")
sh("guarded_eval", "python guarded_eval.py results/verify.json --out results/guarded_eval.json")

## 6. Part 3 — silent-failure probes

In [ ]:
check_same("adapter_main", "runs/main"); check_same("nf4_main", "runs/main-merged-nf4")
sh("silent_failures", "python silent_failures.py --base Qwen/Qwen2.5-1.5B-Instruct --adapter runs/main "
                      "--merged runs/main-merged --nf4 runs/main-merged-nf4 --verify results/verify.json")

## 7. Soup's general suites with every answer recorded (fp16), then Soup's own `ship` gate (bf16)

In [ ]:
check_same("merged_main", "runs/main-merged"); check_same("nf4_main", "runs/main-merged-nf4")
sh("general_probe", "python general_probe.py --base Qwen/Qwen2.5-1.5B-Instruct --tuned runs/main-merged "
                    "--nf4 runs/main-merged-nf4 --out results/general_probe.json")

In [ ]:
B = "--base Qwen/Qwen2.5-1.5B-Instruct --task-eval data/ship_eval.jsonl --device cuda"
check_same("adapter_main", "runs/main")
sh("ship_main", f"soup --no-telemetry ship {B} --adapter runs/main -o results/ship_main.json", check=False)       # exit 2 = DON'T SHIP, expected
check_same("adapter_control", "runs/control")
sh("ship_control", f"soup --no-telemetry ship {B} --adapter runs/control -o results/ship_control.json", check=False)
check_same("nf4_main", "runs/main-merged-nf4")
sh("ship_nf4", f"soup --no-telemetry ship {B} --tuned runs/main-merged-nf4 -o results/ship_nf4.json", check=False)
for f in ("results/ship_main.json", "results/ship_control.json", "results/ship_nf4.json"):
    need(f); print(f, json.load(open(f))["decision"])

## 8. Memory report, throughput, final provenance check, package

In [ ]:
sh("memory_report", "python phase_report.py")
sh("throughput", """python - <<'EOF'
import json, re
tok = json.load(open("data/tokens.json"))
log = open("logs/train_main.log").read()
rt = float(re.findall(r"'train_runtime': '([0-9.]+)'", log)[-1])
real = sum(tok) * 1350 / len(tok) * 2
trained = sum(sum(json.loads(l)["row_tokens"]) for l in open("runs/main/rewind.jsonl") if '"batch"' in l)
print(f"train_runtime {rt:.1f}s | real tokens/s {real/rt:.0f} | loss-bearing tokens/s {trained/rt:.0f} | loss-bearing share {trained/real:.1%}")
EOF""")
for n, d in (("adapter_main","runs/main"),("merged_main","runs/main-merged"),("nf4_main","runs/main-merged-nf4"),
             ("adapter_control","runs/control"),("merged_control","runs/control-merged")):
    check_same(n, d)
print("ALL ARTIFACTS UNCHANGED FROM BIRTH TO LAST RESULT")
# package: Python zipfile, so a missing optional file is listed, not fatal
OUT = f"{WORK}/deliverables_v2.zip"
items = (["logs", "results", "configs", "data/manifest.json"] + sorted(glob.glob("data/*.jsonl")) + sorted(glob.glob("*.py"))
         + sorted(glob.glob("*.ipynb")) + ["runs/main/soup_run.json", "runs/main/rewind.jsonl", "runs/control/rewind.jsonl",
         "runs/main-merged-nf4/quantize_stats.json"])
missing = [i for i in items if not os.path.exists(i)]
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for i in items:
        if os.path.isdir(i):
            for f in sorted(glob.glob(f"{i}/**/*", recursive=True)):
                if os.path.isfile(f): z.write(f)
        elif os.path.exists(i):
            z.write(i)
print(f"wrote {OUT} ({os.path.getsize(OUT)/2**20:.1f} MiB); not found (listed, not packaged): {missing}")
try:                                                     # editor download link; after a commit it is in Output
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(OUT, os.getcwd())))
except ImportError:
    pass